In [ ]:
# Aplica DDL/create-catalog.sql no metastore antes do resto do pipeline rodar.
# CREATE * usa IF NOT EXISTS; os ALTER TABLE ADD COLUMNS nao tem equivalente
# suportado pelo parser do spark.sql(), entao tratamos "coluna ja existe" como
# no-op abaixo para manter tudo idempotente a cada execucao do job.
dbutils.widgets.text("ddl_path", "")
ddl_path = dbutils.widgets.get("ddl_path")

In [ ]:
with open(ddl_path) as f:
    raw_sql = f.read()

sql_without_comments = "\n".join(
    line for line in raw_sql.splitlines() if not line.strip().startswith("--")
)

statements = [stmt.strip() for stmt in sql_without_comments.split(";") if stmt.strip()]

applied, skipped = 0, 0
for statement in statements:
    try:
        spark.sql(statement)
        applied += 1
    except Exception as e:
        if "already exists" in str(e).lower():
            skipped += 1
        else:
            raise

print(f"{applied} statements aplicados, {skipped} ja existentes (skip) -- {ddl_path}")